# ST-CDGM — Validation robuste et inférence stabilisée

Ce notebook charge un checkpoint sauvegardé et reconstruit les modèles pour :
- validation robuste sur les jeux de test par GCM,
- tests de non-régression,
- pipeline d'inférence stabilisée (vérifications de formes / NaN / Inf),
- export d'artefacts (CSV / JSON).

In [1]:
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf

project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Reproductibilite
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

CONFIG = OmegaConf.load("config/training_config.yaml")
DEVICE = torch.device("cuda" if torch.cuda.is_available() and CONFIG.training.device == "cuda" else "cpu")

print(f"Device: {DEVICE}")
print("Config chargee depuis config/training_config.yaml")

Device: cpu
Config chargee depuis config/training_config.yaml


In [2]:
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models.intelligible_encoder import IntelligibleVariableConfig, IntelligibleVariableEncoder
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder
from st_cdgm.evaluation.evaluation_xai import evaluate_metrics, compute_temporal_variance_metrics

# Chemin explicite (optionnel) : None = recherche automatique dans save_dir
USER_CHECKPOINT_PATH = None  # ex: Path("models/st_cdgm_checkpoint_best.pth")
_env_ckpt = os.environ.get("ST_CDGM_CHECKPOINT", "").strip()
if _env_ckpt:
    USER_CHECKPOINT_PATH = Path(_env_ckpt)

_save_dir = Path(CONFIG.checkpoint.get("save_dir", "models"))
_candidates = []
if USER_CHECKPOINT_PATH:
    _candidates.append(Path(USER_CHECKPOINT_PATH))
for name in (
    "st_cdgm_checkpoint_best.pth",
    "st_cdgm_checkpoint.pth",
    "st_cdgm_checkpoint_last.pth",
):
    _candidates.append(_save_dir / name)

CHECKPOINT_PATH = next((p for p in _candidates if p.is_file()), None)
if CHECKPOINT_PATH is None and _save_dir.is_dir():
    _pth = sorted(_save_dir.glob("*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
    if _pth:
        CHECKPOINT_PATH = _pth[0]
        print(f"[INFO] Aucun nom standard — utilisation du .pth le plus recent: {CHECKPOINT_PATH}")

if CHECKPOINT_PATH is None or not CHECKPOINT_PATH.is_file():
    _msg = (
        "Aucun checkpoint PyTorch trouve. Chemins testes :\n  - "
        + "\n  - ".join(str(p) for p in _candidates)
        + (f"\n  - (aucun *.pth dans {_save_dir})" if _save_dir.is_dir() else f"\n  - (dossier absent: {_save_dir})")
        + "\n\nActions possibles :\n"
        "  1. Executer l'entrainement (st_cdgm_training_evaluation.ipynb ou train_ddp.py).\n"
        "  2. Definir la variable d'environnement ST_CDGM_CHECKPOINT=chemin/vers/fichier.pth\n"
        "  3. Dans cette cellule, definir USER_CHECKPOINT_PATH = Path(\"...\") (prioritaire sur la recherche auto)."
    )
    raise FileNotFoundError(_msg)

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
required_keys = {"encoder_state_dict", "rcn_cell_state_dict", "diffusion_state_dict"}
missing_keys = required_keys.difference(checkpoint.keys())
if missing_keys:
    raise KeyError(f"Checkpoint invalide, cles manquantes: {missing_keys}")

print(f"Checkpoint charge: {CHECKPOINT_PATH}")
print(f"Epoch checkpoint: {checkpoint.get('epoch', 'n/a')}")

c:\Users\reall\Desktop\climate_data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\reall\Desktop\climate_data\.venv\Lib\site-packages\torch\amp\autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Checkpoint charge: models\st_cdgm_checkpoint.pth
Epoch checkpoint: 10


In [3]:
# Configuration test (dossiers et GCM)
TEST_ROOT = Path("data/raw/test")
if not TEST_ROOT.exists():
    raise FileNotFoundError(f"Repertoire test introuvable: {TEST_ROOT}")

def test_lr_path(gcm):
    return str(TEST_ROOT / f"{gcm}_histupdated_compressed.nc")

def test_hr_path(gcm):
    return str(TEST_ROOT / f"{gcm}_historical_precip_compressed.nc")

def discover_test_gcms():
    lr_suffix = "_histupdated_compressed.nc"
    hr_suffix = "_historical_precip_compressed.nc"
    lr_stems = {f.name.replace(lr_suffix, "") for f in TEST_ROOT.glob(f"*{lr_suffix}")}
    hr_stems = {f.name.replace(hr_suffix, "") for f in TEST_ROOT.glob(f"*{hr_suffix}")}
    return sorted(lr_stems.intersection(hr_stems))

TEST_GCMS = discover_test_gcms()
if not TEST_GCMS:
    raise RuntimeError("Aucun couple LR/HR test detecte dans data/raw/test.")

print(f"GCMs detectes: {TEST_GCMS}")

GCMs detectes: ['EC-Earth3', 'NorESM2-MM']


In [4]:
# Rebuild pipeline + builder + modeles depuis checkpoint
ckpt_cfg = checkpoint.get("config", {})
ckpt_cfg_full = checkpoint.get("config_full", {})

SEQ_LEN = int(ckpt_cfg.get("seq_len", CONFIG.data.seq_len))
STATIC_PATH = str(CONFIG.data.static_path) if CONFIG.data.get("static_path") else None

norm_cfg = ckpt_cfg.get("normalization", {})
MEAN_PATH = str(norm_cfg.get("mean_path", "data/raw/normalization_coefs/mean_1974_2011.nc"))
STD_PATH = str(norm_cfg.get("std_path", "data/raw/normalization_coefs/std_1974_2011.nc"))

LR_VARIABLES = list(ckpt_cfg.get("lr_variables", CONFIG.data.lr_variables))
STATIC_VARIABLES = list(ckpt_cfg.get("static_variables", CONFIG.data.static_variables))

# Convention test downscaling: cible toujours pr
TEST_HR_VARIABLES = ["pr"]

sampling_cfg = ckpt_cfg.get("sampling", {})
EVAL_NUM_STEPS = int(sampling_cfg.get("num_steps", CONFIG.diffusion.get("val_num_steps", 15)))
EVAL_SCHEDULER = sampling_cfg.get("scheduler_type", CONFIG.diffusion.get("scheduler_type", "ddpm"))

# Taille d'ensemble pour des metriques probabilistes (CRPS/Wasserstein/Energy score)
EVAL_NUM_SAMPLES = 5

def build_test_pipeline(gcm):
    return NetCDFDataPipeline(
        lr_path=test_lr_path(gcm),
        hr_path=test_hr_path(gcm),
        static_path=STATIC_PATH,
        seq_len=SEQ_LEN,
        baseline_strategy=CONFIG.data.baseline_strategy,
        baseline_factor=CONFIG.data.get("baseline_factor", 4),
        normalize=bool(CONFIG.data.normalize),
        nan_fill_strategy=CONFIG.data.nan_fill_strategy,
        precipitation_delta=CONFIG.data.get("precipitation_delta", 0.01),
        lr_variables=LR_VARIABLES,
        hr_variables=TEST_HR_VARIABLES,
        static_variables=STATIC_VARIABLES,
        means_path=MEAN_PATH if Path(MEAN_PATH).exists() else None,
        stds_path=STD_PATH if Path(STD_PATH).exists() else None,
    )

probe_gcm = TEST_GCMS[0]
probe_pipeline = build_test_pipeline(probe_gcm)
probe_dataset = probe_pipeline.build_sequence_dataset(seq_len=SEQ_LEN, as_torch=True)
probe_sample = next(iter(probe_dataset))

lr_shape = tuple(ckpt_cfg.get("lr_shape", CONFIG.graph.lr_shape))
hr_shape = tuple(ckpt_cfg.get("hr_shape", CONFIG.graph.hr_shape))
include_mid_layer = bool(ckpt_cfg_full.get("graph", {}).get("include_mid_layer", CONFIG.graph.include_mid_layer))

builder = HeteroGraphBuilder(
    lr_shape=lr_shape,
    hr_shape=hr_shape,
    static_dataset=probe_pipeline.get_static_dataset(),
    include_mid_layer=include_mid_layer,
)

allowed_nodes = set(builder.dynamic_node_types) | set(builder.static_node_types)
encoder_configs = [
    IntelligibleVariableConfig(
        name=mp.name,
        meta_path=(mp.src, mp.relation, mp.target),
        pool=mp.get("pool", "mean"),
    )
    for mp in CONFIG.encoder.metapaths
    if mp.src in allowed_nodes and mp.target in allowed_nodes
]
if probe_pipeline.get_static_dataset() is not None:
    encoder_configs.append(
        IntelligibleVariableConfig(
            name="static",
            meta_path=("SP_HR", "causes", "GP850"),
            pool="mean",
        )
    )

hr_channels = probe_sample["residual"].shape[1]
rcn_driver_dim = probe_sample["lr"].shape[1]

encoder = IntelligibleVariableEncoder(
    configs=encoder_configs,
    hidden_dim=CONFIG.encoder.hidden_dim,
    conditioning_dim=CONFIG.encoder.conditioning_dim,
).to(DEVICE)

rcn_cell = RCNCell(
    num_vars=len(encoder_configs),
    hidden_dim=CONFIG.rcn.hidden_dim,
    driver_dim=rcn_driver_dim,
    reconstruction_dim=rcn_driver_dim,
    dropout=CONFIG.rcn.dropout,
).to(DEVICE)

rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))

diffusion = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=CONFIG.diffusion.height,
    width=CONFIG.diffusion.width,
    num_diffusion_steps=CONFIG.diffusion.steps,
    unet_kwargs=dict(
        layers_per_block=1,
        block_out_channels=(32, 64),
        down_block_types=("DownBlock2D", "CrossAttnDownBlock2D"),
        up_block_types=("CrossAttnUpBlock2D", "UpBlock2D"),
        mid_block_type="UNetMidBlock2D",
        norm_num_groups=8,
        class_embed_type="projection",
        projection_class_embeddings_input_dim=len(encoder_configs) * CONFIG.diffusion.conditioning_dim,
        resnet_time_scale_shift="scale_shift",
        attention_head_dim=32,
        only_cross_attention=[False, True],
    ),
).to(DEVICE)

try:
    from st_cdgm.utils.checkpoint import strip_torch_compile_prefix
except ModuleNotFoundError:
    def strip_torch_compile_prefix(sd):
        if not sd:
            return sd
        if not any(str(k).startswith("_orig_mod.") for k in sd.keys()):
            return sd
        return {
            (k[len("_orig_mod.") :] if str(k).startswith("_orig_mod.") else k): v
            for k, v in sd.items()
        }

encoder.load_state_dict(strip_torch_compile_prefix(checkpoint["encoder_state_dict"]))
rcn_cell.load_state_dict(strip_torch_compile_prefix(checkpoint["rcn_cell_state_dict"]))
diffusion.load_state_dict(strip_torch_compile_prefix(checkpoint["diffusion_state_dict"]))

encoder.eval()
rcn_runner.cell.eval()
diffusion.eval()

print("Modeles reconstruits et poids charges depuis checkpoint.")
print(f"Sampling eval: steps={EVAL_NUM_STEPS}, scheduler={EVAL_SCHEDULER}")

Modeles reconstruits et poids charges depuis checkpoint.
Sampling eval: steps=15, scheduler=ddpm


c:\Users\reall\Desktop\climate_data\src\st_cdgm\models\intelligible_encoder.py:92: UserWarning: There exist node types ({'SP_HR'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  self.hetero_conv = HeteroConv(convs_dict, aggr="sum")


In [5]:
def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]

    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)

    dynamic_features = {node_type: lr_nodes_steps[0] for node_type in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)

    out = {
        "lr": lr_tensor,
        "residual": sample["residual"],
        "baseline": sample.get("baseline"),
        "hetero": hetero,
    }
    if "valid_mask" in sample:
        out["valid_mask"] = sample["valid_mask"]
    return out


def extract_target_and_baseline_t_mean(batch, device):
    """Converit ground-truth en format compatible avec `evaluate_metrics`.

    Le module évalue sur `DiffusionOutput.t_mean`, qui correspond au champ HR reconstruit.
    Ici, le dataset fournit `residual` et `baseline`, donc `t_mean ~= baseline + residual`.
    """
    target_residual = batch["residual"][-1].to(device)
    baseline_tensor = (
        batch["baseline"][-1] if batch["baseline"] is not None else torch.zeros_like(target_residual)
    )
    baseline_tensor = baseline_tensor.to(device)

    full_target = baseline_tensor + target_residual

    baseline_tensor = torch.nan_to_num(baseline_tensor, nan=0.0, posinf=0.0, neginf=0.0)
    full_target = torch.nan_to_num(full_target, nan=0.0, posinf=0.0, neginf=0.0)

    baseline_batch = baseline_tensor.unsqueeze(0) if baseline_tensor.dim() == 3 else baseline_tensor
    target_batch = full_target.unsqueeze(0) if full_target.dim() == 3 else full_target

    return target_batch, baseline_batch


def _max_pool_4x4(field):
    """Agrège un champ [B,C,H,W] ou [C,H,W] par blocs 4x4 (max)."""
    if field.dim() == 3:
        field = field.unsqueeze(0)
    if field.dim() != 4:
        raise ValueError(f"Forme inattendue pour max4 pooling: {tuple(field.shape)}")
    return F.max_pool2d(field, kernel_size=4, stride=4)


def compute_max4_crps(samples_out, target_batch):
    """CRPS calcule apres agrégation spatiale max en blocs 4x4."""
    pooled_samples = [_max_pool_4x4(s.t_mean) for s in samples_out]
    pooled_target = _max_pool_4x4(target_batch)
    stack = torch.stack(pooled_samples, dim=0)
    term1 = torch.abs(stack - pooled_target.unsqueeze(0)).mean(dim=0)
    pairwise = torch.abs(stack.unsqueeze(0) - stack.unsqueeze(1)).mean(dim=(0, 1))
    crps = (term1 - 0.5 * pairwise).mean().item()
    return float(crps)


def compute_sigma_r(pred_fields, target_fields, eps=1e-8):
    """Ratio de variabilité temporelle: sigma_r = std(pred)/std(target)."""
    if len(pred_fields) < 2 or len(target_fields) < 2:
        return {"sigma_r": float("nan"), "sigma_r_pct": float("nan")}
    pred_stack = torch.stack([p.squeeze(0) if p.dim() == 4 else p for p in pred_fields], dim=0)
    target_stack = torch.stack([t.squeeze(0) if t.dim() == 4 else t for t in target_fields], dim=0)
    std_pred = pred_stack.std(dim=0).mean().item()
    std_target = target_stack.std(dim=0).mean().item()
    if std_target <= eps:
        return {"sigma_r": float("nan"), "sigma_r_pct": float("nan")}
    sigma_r = std_pred / (std_target + eps)
    return {"sigma_r": float(sigma_r), "sigma_r_pct": float(100.0 * sigma_r)}


@torch.no_grad()
def generate_prediction_stable(sample):
    batch = convert_sample_to_batch(sample, builder, DEVICE)

    lr_data = batch["lr"].to(DEVICE)
    target_batch, baseline_batch = extract_target_and_baseline_t_mean(batch, DEVICE)

    H_init = encoder.init_state(batch["hetero"]).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_output = rcn_runner.run(H_init, drivers, reconstruction_sources=drivers)

    conditioning = encoder.project_state_tensor(seq_output.states[-1]).to(DEVICE)

    samples_out = []
    for _ in range(EVAL_NUM_SAMPLES):
        generated = diffusion.sample(
            conditioning,
            num_steps=EVAL_NUM_STEPS,
            scheduler_type=EVAL_SCHEDULER,
            apply_constraints=False,
            baseline=baseline_batch,
        )

        # Garantit une forme spatiale compatible avec la target
        if generated.t_mean.shape != target_batch.shape:
            generated.t_mean = F.interpolate(
                generated.t_mean,
                size=target_batch.shape[-2:],
                mode="bicubic",
                align_corners=False,
            ).clamp(min=0.0)

        generated.t_mean = torch.nan_to_num(generated.t_mean, nan=0.0, posinf=0.0, neginf=0.0)

        # Met en CPU pour reduire la pression memoire
        generated.residual = generated.residual.cpu()
        if generated.baseline is not None:
            generated.baseline = generated.baseline.cpu()
        generated.t_min = generated.t_min.cpu()
        generated.t_mean = generated.t_mean.cpu()
        generated.t_max = generated.t_max.cpu()

        samples_out.append(generated)

    return samples_out, target_batch.cpu(), baseline_batch.cpu()


def compute_metrics_basic(full_pred, full_target):
    mask = ~torch.isnan(full_pred) & ~torch.isnan(full_target)
    if mask.sum() == 0:
        return {"MSE": float("nan"), "MAE": float("nan"), "Correlation": float("nan")}

    p = full_pred[mask].float()
    t = full_target[mask].float()

    mse = torch.mean((p - t) ** 2).item()
    mae = torch.mean(torch.abs(p - t)).item()

    p_c = p - p.mean()
    t_c = t - t.mean()
    eps = 1e-8
    denom = torch.sqrt((p_c ** 2).sum()) * torch.sqrt((t_c ** 2).sum()) + eps
    corr = ((p_c * t_c).sum() / denom).item() if denom.item() > eps else 0.0

    return {"MSE": mse, "MAE": mae, "Correlation": corr}

print("Inference stabilisee et metriques de base pretes.")

Inference stabilisee et metriques de base pretes.


In [6]:
# Evaluation robuste complete
total_t0 = time.perf_counter()
if CONFIG.get("evaluation"):
    _max_test_samples = CONFIG.evaluation.get("max_test_samples", None)
else:
    _max_test_samples = None

# Plafond dur pour limiter le temps tout en gardant une eval fiable
if _max_test_samples is None:
    N_TEST_SAMPLES = 10
else:
    try:
        N_TEST_SAMPLES = int(_max_test_samples)
    except Exception:
        N_TEST_SAMPLES = 10
    if N_TEST_SAMPLES <= 0:
        N_TEST_SAMPLES = 10
    N_TEST_SAMPLES = min(N_TEST_SAMPLES, 10)

rows = []
gcm_timing = {}
for gcm in TEST_GCMS:
    print(f"\n=== Evaluation GCM: {gcm} ===")
    gcm_t0 = time.perf_counter()
    pipe_test = build_test_pipeline(gcm)
    test_dataset = pipe_test.build_sequence_dataset(seq_len=SEQ_LEN, as_torch=True)

    iterator = iter(test_dataset)
    n_done = 0
    while True:
        if N_TEST_SAMPLES is not None and n_done >= N_TEST_SAMPLES:
            break
        try:
            sample = next(iterator)
        except StopIteration:
            break

        sample_t0 = time.perf_counter()
        samples_out, target_batch, baseline_batch = generate_prediction_stable(sample)
        sample_inference_seconds = float(time.perf_counter() - sample_t0)

        pred_mean_t_mean = torch.stack([s.t_mean for s in samples_out], dim=0).mean(dim=0)
        pred_mean_t_mean = torch.nan_to_num(pred_mean_t_mean, nan=0.0, posinf=0.0, neginf=0.0)

        f1_percentiles = list(CONFIG.evaluation.get("f1_percentiles", [95.0, 99.0])) if CONFIG.get("evaluation") else [95.0, 99.0]

        metrics_report = evaluate_metrics(
            samples_out,
            target_batch,
            baseline=baseline_batch,
            compute_advanced=True,
            f1_percentiles=f1_percentiles,
        )

        corr = compute_metrics_basic(pred_mean_t_mean, target_batch).get("Correlation", float("nan"))
        m = {
            "MSE": metrics_report.mse,
            "MAE": metrics_report.mae,
            "Correlation": corr,
            "hist_distance": metrics_report.hist_distance,
            "crps": metrics_report.crps,
            "spectrum_distance": metrics_report.spectrum_distance,
            "baseline_mse": metrics_report.baseline_mse,
            "baseline_mae": metrics_report.baseline_mae,
            "fss": metrics_report.fss,
            "wasserstein_distance": metrics_report.wasserstein_distance,
            "energy_score": metrics_report.energy_score,
        }

        if metrics_report.f1_extremes:
            for k, v in metrics_report.f1_extremes.items():
                m[f"F1_{k}"] = float(v)

        preds_for_var = [s.t_mean for s in samples_out]
        targets_for_var = [target_batch for _ in range(len(samples_out))]
        m.update(compute_temporal_variance_metrics(preds_for_var, targets_for_var))
        m.update(compute_sigma_r(preds_for_var, targets_for_var))
        m["max4_crps"] = compute_max4_crps(samples_out, target_batch)
        m["sample_inference_seconds"] = sample_inference_seconds
        m["gcm"] = gcm
        m["sample_idx"] = n_done
        rows.append(m)
        n_done += 1

    gcm_inference_seconds = float(time.perf_counter() - gcm_t0)
    gcm_timing[gcm] = gcm_inference_seconds
    print(f"Echantillons evalues: {n_done}")
    print(f"Temps inference GCM ({gcm}): {gcm_inference_seconds:.2f}s")

if not rows:
    raise RuntimeError("Aucune metrique calculee. Verifiez les donnees test/chemins/checkpoint.")

total_inference_seconds = float(time.perf_counter() - total_t0)

metrics_df = pd.DataFrame(rows)
gcm_summary_df = metrics_df.groupby("gcm", as_index=False).mean(numeric_only=True)
global_summary = metrics_df.mean(numeric_only=True).to_dict()

print("\nResume global:")
for k in [
    "MSE",
    "MAE",
    "Correlation",
    "temporal_var_rmse",
    "temporal_var_corr",
    "sigma_r",
    "sigma_r_pct",
    "max4_crps",
    "sample_inference_seconds",
    "crps",
    "energy_score",
    "wasserstein_distance",
    "spectrum_distance",
    "F1_p95",
    "F1_p99",
]:
    print(f"  - {k}: {global_summary.get(k, float('nan')):.6f}")
print(f"  - total_inference_seconds: {total_inference_seconds:.6f}")

gcm_summary_df


=== Evaluation GCM: EC-Earth3 ===


MemoryError: Unable to allocate 857. MiB for an array with shape (179, 1255600) and data type float32

In [ ]:
# Non-regression + export artefacts
RESULTS_DIR = Path("results/validation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Seuils par defaut (ajustables)
DEFAULT_THRESHOLDS = {
    "MSE_MAX": float(CONFIG.evaluation.get("mse_max", 0.05)) if CONFIG.get("evaluation") else 0.05,
    "MAE_MAX": float(CONFIG.evaluation.get("mae_max", 0.2)) if CONFIG.get("evaluation") else 0.2,
    "CORR_MIN": float(CONFIG.evaluation.get("corr_min", 0.1)) if CONFIG.get("evaluation") else 0.1,
}

mse_ok = global_summary.get("MSE", float("inf")) <= DEFAULT_THRESHOLDS["MSE_MAX"]
mae_ok = global_summary.get("MAE", float("inf")) <= DEFAULT_THRESHOLDS["MAE_MAX"]
corr_ok = global_summary.get("Correlation", float("-inf")) >= DEFAULT_THRESHOLDS["CORR_MIN"]

non_reg_pass = bool(mse_ok and mae_ok and corr_ok)

# Parametres d'entrainement (si disponibles)
_loss_cfg_ckpt = checkpoint.get("config_full", {}).get("loss", {}) if isinstance(checkpoint, dict) else {}
_loss_cfg_runtime = dict(CONFIG.loss) if CONFIG.get("loss") else {}
_loss_cfg = {**_loss_cfg_runtime, **_loss_cfg_ckpt}

lambda_adv = _loss_cfg.get("lambda_adv", _loss_cfg.get("ad_loss_factor", _loss_cfg.get("adv_weight", "N/A")))
if lambda_adv is not None and lambda_adv != "N/A":
    try:
        lambda_adv = float(lambda_adv)
    except Exception:
        lambda_adv = str(lambda_adv)

intensity_constraint_enabled = _loss_cfg.get(
    "intensity_constraint_enabled",
    _loss_cfg.get("use_intensity_constraint", _loss_cfg.get("ic_enabled", "N/A")),
)
if intensity_constraint_enabled not in ("N/A", None):
    intensity_constraint_enabled = bool(intensity_constraint_enabled)

report = {
    "checkpoint_path": str(CHECKPOINT_PATH),
    "sampling": {
        "eval_num_steps": int(EVAL_NUM_STEPS),
        "eval_scheduler": str(EVAL_SCHEDULER),
        "eval_num_samples": int(EVAL_NUM_SAMPLES),
        "max_test_samples_per_gcm": int(N_TEST_SAMPLES),
    },
    "timing": {
        "total_inference_seconds": float(total_inference_seconds),
        "gcm_inference_seconds": {k: float(v) for k, v in gcm_timing.items()},
        "mean_sample_inference_seconds": float(metrics_df["sample_inference_seconds"].mean()) if "sample_inference_seconds" in metrics_df else float("nan"),
    },
    "training_recipe": {
        "lambda_adv": lambda_adv,
        "intensity_constraint_enabled": intensity_constraint_enabled,
    },
    "n_samples": int(len(metrics_df)),
    "thresholds": DEFAULT_THRESHOLDS,
    "global_summary": {k: float(v) for k, v in global_summary.items()},
    "checks": {
        "mse_ok": mse_ok,
        "mae_ok": mae_ok,
        "corr_ok": corr_ok,
    },
    "non_regression_pass": non_reg_pass,
}

metrics_path = RESULTS_DIR / "validation_metrics_by_sample.csv"
gcm_path = RESULTS_DIR / "validation_metrics_by_gcm.csv"
report_path = RESULTS_DIR / "validation_report.json"

metrics_df.to_csv(metrics_path, index=False)
gcm_summary_df.to_csv(gcm_path, index=False)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(f"\nVerdict non-regression: {'PASS' if non_reg_pass else 'FAIL'}")
print(f"Timing total: {total_inference_seconds:.2f}s")
if "sample_inference_seconds" in metrics_df:
    print(f"Timing moyen/echantillon: {metrics_df['sample_inference_seconds'].mean():.2f}s")
print(f"Recette entrainement -> lambda_adv: {lambda_adv}, IC: {intensity_constraint_enabled}")
print(f"CSV sample-level: {metrics_path}")
print(f"CSV GCM-level: {gcm_path}")
print(f"JSON report: {report_path}")

report